# Chapter 5 — Methods and Methodologies
### Notebook 3 · Exercises

*Book reference: Section 5.3*

The book's exercises, executable. Assertions are the marking scheme.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch05_toolkit as ch5
from oe_course.sparql import SparqlStore
from oe_course.data import corpus
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

### Exercise R1 — Match methodologies to projects

For each of four project descriptions, recommend a methodology and justify it with the matched signals.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution R1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
projects = {
    'A hospital wants to integrate two legacy databases and an existing thesaurus.':
        'neon',
    'A university lab is building a new ontology from scratch, single team, full lifecycle.':
        'methontology',
    'Twenty labs across Europe will jointly edit an evolving, decentralised ontology.':
        'diligent',
    'A startup wants small increments with a test driven, agile, iterative process.':
        'samod',
}
for brief, expected in projects.items():
    r = ch5.recommend_methodology(brief)
    print(f"{r['recommended']:16s} (expected {expected:16s}) <- {r['reason']}")
    assert r['recommended'] == expected
print('\nNote the recommender reads SIGNALS, not domains. "Hospital" is\n'
      'irrelevant; "existing thesaurus" is decisive.')

### Exercise R2 — Turn three requirements into competency questions

Write SPARQL for three requirements over the AWO, and report the coverage.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution R2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
mine = [
    ch5.CompetencyQuestion('m1', 'Which classes are carnivores?',
                           'SELECT ?c WHERE { ?c rdfs:subClassOf awo:Carnivore }'),
    ch5.CompetencyQuestion('m2', 'Which object properties exist?',
                           'SELECT ?p WHERE { ?p a owl:ObjectProperty }'),
    ch5.CompetencyQuestion('m3', 'Which properties are transitive?',
                           'SELECT ?p WHERE { ?p a owl:TransitiveProperty }'),
]
store = SparqlStore.in_memory(corpus.get('awo').turtle)
result = ch5.cq_coverage(mine, store)
print(pd.DataFrame(result['results']).to_string(index=False))
print(f"coverage {result['coverage']}")
assert result['coverage'] == 1.0

### Exercise R3 — Find the OntoClean error in a realistic taxonomy

Audit this plausible-looking taxonomy and explain the error in terms a domain expert would accept.

In [ ]:
taxonomy = [('Customer', 'Person'), ('Person', 'Customer'), ('Person', 'Agent')]
# YOUR CODE HERE


<details>
<summary>Solution R3</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
tags = dict(ch5.ONTOCLEAN_TAGS)
tags['Customer'] = ch5.MetaProperties('~R', '-I', '-U', '+D')
tags['Agent'] = ch5.MetaProperties('+R', '+I', '+U', '-D')

taxonomy = [('Customer', 'Person'), ('Person', 'Customer'), ('Person', 'Agent')]
violations = ch5.ontoclean_violations(taxonomy, tags)
for v in violations:
    print(f"[{v['constraint']}] {v['axiom']}: {v['detail']}")
assert any(v['axiom'] == 'Person <= Customer' for v in violations)
print('\nFor a domain expert: "every person is a customer" would mean a person\n'
      'stops being a person the moment they stop buying from us. The taxonomy\n'
      'says something about our database that is not true about the world -- the\n'
      'classic mistake of modelling the application instead of the domain.')

## Where this leaves you

You can select a methodology from evidence, express requirements as tests that can fail, and detect a class of error that survives every reasoner in the course so far. Notebook 4 hands all of it to an agent — and asks what happens when the reward model for a *plan* is subtly wrong.